# 🇩🇪 LernSathi — Your German AI Tutor · Google Colab Edition

This notebook runs your **entire local AI-tutor codebase** in the cloud — nothing is changed, every file below is an exact copy of the repo:

| Stage | Model | Local file |
|---|---|---|
| 🎤 Speech → Text | OpenAI **Whisper `small`** (German) | `ai/speech/stt.py` |
| 💬 Conversation | **Qwen3 1.7B** via **Ollama** | `ai/llm/model.py` |
| 🔊 Text → Speech | **Piper** · Thorsten (de, medium) | `ai/speech/tts.py` |
| 🖥️ UI | **Streamlit** chat + browser mic | `ui/`, `app.py` |
| 🔗 Glue | `ConversationService` | `services/conversation_service.py` |

### How to use
1. **Runtime ▸ Change runtime type ▸ T4 GPU** *(recommended — works CPU-only too, just slower)*
2. **Runtime ▸ Run all** — total ≈ 8–12 min (most of it is one-time model downloads)
3. At the end you get a public **`https://…trycloudflare.com`** link → open it on any device, **allow the microphone**, pick your German level (**A1–C2**), and start speaking German 🗣️

> The Streamlit app streams through an HTTPS Cloudflare tunnel, so **voice input (microphone) works**, exactly like running locally.

In [ ]:
#@title Step 1 — System setup: Ollama · ffmpeg · cloudflared

# Install required system packages FIRST
!apt-get -qq update
!apt-get -qq install -y zstd ffmpeg

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Verify installations
!echo "--- installed ---"
!ollama --version
!ffmpeg -version 2>/dev/null | head -n 1
!cloudflared --version

In [ ]:
# Start Ollama server in the background
!nohup ollama serve > /tmp/ollama.log 2>&1 &

# Give it a few seconds to start
!sleep 5

# Check the API
!curl http://127.0.0.1:11434/api/tags 

In [ ]:
#@title Step 2 — Install Python packages (~2 min)
# NOTE: this installs OpenAI's `openai-whisper`.
# (Do NOT `pip install whisper` — that's an unrelated time-series package.)
%pip install -q streamlit openai-whisper "piper-tts==1.7.0" ollama

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
#@title Step 3 — Recreate the project structure (`/content` = project root)
import pathlib

DIRS = [
    "ai/llm",
    "ai/speech",
    "services",
    "ui/mic_frontend",
    "models/tts",
    "audio/input",
    "audio/output",
]
FILES = [
    "ai/__init__.py",
    "ai/llm/__init__.py",
    "ai/speech/__init__.py",
    "services/__init__.py",
    "ui/__init__.py",
    "ai/llm/model.py",
    "ai/speech/stt.py",
    "ai/speech/tts.py",
    "services/conversation_service.py",
    "ui/chat.py",
    "ui/mic_widget.py",
    "ui/mic_frontend/index.html",
]

for d in DIRS:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
for f in FILES:
    pathlib.Path(f).touch(exist_ok=True)

print("Project tree ready ✔")

## 📦 The codebase (exact copies of your repo)

Every `%%writefile` cell below writes one file from your local project, byte-for-byte.

### 🧠 AI layer — STT / LLM / TTS

In [ ]:
%%writefile ai/llm/model.py
import ollama


_BASE_PROMPT = """
Du bist ein freundlicher und geduldiger deutscher Sprachlehrer.

Deine Aufgabe ist es, mit dem Benutzer auf Deutsch zu sprechen
und ihm dabei zu helfen, sein Deutsch zu verbessern.

Allgemeine Regeln:
- Sprich hauptsächlich auf Deutsch.
- Halte deine Antworten natürlich und nicht unnötig lang.
- Stelle gelegentlich Fragen, damit das Gespräch weitergeht.
- Wenn der Benutzer einen grammatikalischen Fehler macht,
  korrigiere ihn freundlich.
"""

LEVEL_PROMPTS = {
    "A1": _BASE_PROMPT + """
Das aktuelle Sprachniveau des Benutzers ist A1 (Anfänger).

Regeln für A1:
- Verwende nur SEHR einfache Wörter und kurze Sätze
  (höchstens 8 Wörter pro Satz).
- Benutze fast ausschließlich das Präsens.
- Sprich über einfache Themen: Begrüßung, Familie, Essen,
  Hobbys, Zahlen, Farben.
- Stelle einfache Ja/Nein- oder kurze W-Fragen.
- Erkläre Korrekturen kurz und einfach auf Englisch.
- Schreibe nie mehr als 2–3 kurze Sätze pro Antwort.
""",
    "A2": _BASE_PROMPT + """
Das aktuelle Sprachniveau des Benutzers ist A2 (Grundlagen).

Regeln für A2:
- Verwende häufige Alltagswörter und einfache Sätze.
- Benutze Präsens, Perfekt und das Präteritum von
  sein/haben/modalen Verben.
- Sprich über Alltagsthemen: Einkaufen, Reisen, Wetter,
  Arbeit, Termine, Wohnung.
- Baue einfache Nebensätze mit „weil", „dass" oder „wenn" ein.
- Erkläre Korrekturen kurz auf Englisch oder einfachem Deutsch.
- Schreibe höchstens 3–4 Sätze pro Antwort.
""",
    "B1": _BASE_PROMPT + """
Das aktuelle Sprachniveau des Benutzers ist B1 (Mittelstufe).

Regeln für B1:
- Sprich natürlich über Meinungen, Erfahrungen, Pläne und Träume.
- Benutze komplexere Strukturen: Nebensätze, Wechselpräpositionen
  und den Konjunktiv II für Höflichkeit.
- Führe gelegentlich neue, nützliche Wörter ein.
- Erkläre Korrekturen überwiegend auf Deutsch, bei Bedarf auf Englisch.
- Schreibe höchstens 4–5 Sätze pro Antwort.
""",
    "B2": _BASE_PROMPT + """
Das aktuelle Sprachniveau des Benutzers ist B2 (Fortgeschritten).

Regeln für B2:
- Diskutiere abstrakte und komplexe Themen: Medien, Umwelt,
  Kultur, Beruf, Gesellschaft.
- Argumentiere klar und benutze Passiv, Konjunktiv II und
  komplexere Satzstrukturen.
- Führe idiomatische Ausdrücke ein und erkläre sie kurz.
- Gib präzises Grammatik-Feedback auf Deutsch.
- Schreibe höchstens 5–6 Sätze pro Antwort.
""",
    "C1": _BASE_PROMPT + """
Das aktuelle Sprachniveau des Benutzers ist C1 (sehr fortgeschritten).

Regeln für C1:
- Führe nuancierte Gespräche über anspruchsvolle Themen:
  Gesellschaft, Wissenschaft, Politik, Beruf, Kunst.
- Benutze eine reiche, idiomatische Ausdrucksweise und
  stilistische Varianten.
- Gib differenziertes Feedback zu Stil, Register und Nuancen.
- Antworte ausschließlich auf Deutsch.
- Halte die Antworten fließend und natürlich wie im echten Leben.
""",
    "C2": _BASE_PROMPT + """
Das aktuelle Sprachniveau des Benutzers ist C2 (fast muttersprachlich).

Regeln für C2:
- Sprich wie unter Muttersprachlern: rhetorisch gewandt,
  mit Ironie, Wortspielen und feinen Nuancen, wo es passt.
- Benutze anspruchsvolles Vokabular und Fachsprache passend zum Thema.
- Gib Feedback wie ein Muttersprachler: Stil, Register, Klang.
- Antworte ausschließlich auf Deutsch.
- Sei anspruchsvoll, aber immer respektvoll und ermutigend.
""",
}


class GermanChatbot:

    def __init__(self, model_name: str = "qwen3:4b", level: str = "A1"):
        self.model_name = model_name

        if level not in LEVEL_PROMPTS:
            raise ValueError(
                f"Unknown level '{level}'. "
                f"Choose one of: {', '.join(LEVEL_PROMPTS)}"
            )
        self.level = level

    @property
    def system_prompt(self) -> str:
        return LEVEL_PROMPTS[self.level]

    def set_level(self, level: str):
        if level not in LEVEL_PROMPTS:
            raise ValueError(
                f"Unknown level '{level}'. "
                f"Choose one of: {', '.join(LEVEL_PROMPTS)}"
            )
        self.level = level

    def generate_response(self, messages: list[dict]) -> str:

        conversation = [
            {
                "role": "system",
                "content": self.system_prompt
            }
        ]

        conversation.extend(messages)

        response = ollama.chat(
            model=self.model_name,
            messages=conversation
        )

        return response["message"]["content"].strip()


In [ ]:
%%writefile ai/speech/stt.py
import torch
import whisper

class SpeechToText:
    def __init__(self, model_name: str = "small"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        self.model = whisper.load_model(
            model_name,
            device=self.device
        )

    def transcribe(self, audio_path:str ) -> str:
        result = self.model.transcribe(
            audio_path,
            language="de",
            fp16=self.device == "cuda"
        )  

        text = result["text"]
        return text.strip()

In [ ]:
%%writefile ai/speech/tts.py
import re
from pathlib import Path
import subprocess
import sys
import uuid

EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002500-\U00002BEF"
    "\U00002702-\U000027B0"
    "\U000024C2-\U0001F251"
"]",
    flags=re.UNICODE,
)

def strip_emoji(text: str) -> str:
    return EMOJI_PATTERN.sub("", text)

class TextToSpeech:
    def __init__(self, model_path=None):
        project_root = Path(__file__).resolve().parents[2]

        self.project_root = project_root

        if model_path is None:
            model_path = (
                project_root
                / "models"
                / "tts"
                / "de_DE-thorsten-medium.onnx"
            )

        self.model_path = Path(model_path)

        if not self.model_path.exists():
            raise FileNotFoundError(
                f"TTS model not found: {self.model_path}"
            )

    def synthesize(
        self,
        text: str,
        output_path=None,
    ) -> str:

        text = strip_emoji(text)

        if output_path is None:
            filename = f"response_{uuid.uuid4().hex}.wav"
            output = (
                self.project_root
                / "audio"
                / "output"
                / filename
            )
        else:
            output = Path(output_path)

            # If a relative path is supplied, make it relative
            # to the project root rather than tests/
            if not output.is_absolute():
                output = self.project_root / output

        output.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "piper",
                "--model",
                str(self.model_path),
                "--output_file",
                str(output),
            ],
            input=text.encode("utf-8"),
            capture_output=True,
        )

        if result.returncode != 0:
            error = result.stderr.decode(
                "utf-8",
                errors="replace",
            )

            raise RuntimeError(
                f"Piper TTS failed:\n{error}"
            )

        return str(output)

### 🔗 Service layer — the conversation pipeline

In [ ]:
%%writefile services/conversation_service.py
from pathlib import Path

from ai.speech.stt import SpeechToText
from ai.speech.tts import TextToSpeech
from ai.llm.model import GermanChatbot


class ConversationService:

    def __init__(self, level: str = "A1"):
        print("Loading AI models...")

        self.stt = SpeechToText("small")
        self.chatbot = GermanChatbot(model_name="qwen3:1.7b", level=level)
        self.tts = TextToSpeech()

        print("All AI models loaded.")

    @property
    def level(self) -> str:
        return self.chatbot.level

    def set_level(self, level: str):
        """Switch tutor level without reloading any models."""
        self.chatbot.set_level(level)

    def process_text(
        self,
        user_text: str,
        conversation_history: list[dict],
    ) -> dict:

        if not user_text.strip():
            raise ValueError("Please enter some text.")

        # Generate AI response using conversation_history as context
        # (read-only; service does not modify session state)
        ai_response = self.chatbot.generate_response(conversation_history)

        # Synthesize TTS
        audio_output = self.tts.synthesize(ai_response)

        return {
            "ai_response": ai_response,
            "audio_path": str(audio_output),
        }

    def process_audio(
        self,
        audio_path: str,
        conversation_history: list[dict],
    ) -> dict:

        # 1. Speech → Text
        user_text = self.stt.transcribe(audio_path)

        if not user_text:
            raise ValueError("Could not understand the audio.")

        # 2. Generate AI response using conversation_history as context
        ai_response = self.chatbot.generate_response(conversation_history)

        # 3. Text → Speech
        audio_output = self.tts.synthesize(ai_response)

        # 4. Return everything
        return {
            "user_text": user_text,
            "ai_response": ai_response,
            "audio_path": str(audio_output),
        }

    # --------------------------------------------------
    # Staged methods (used by the UI for step-by-step feedback)
    # TEXT never touches STT; VOICE always goes through Whisper.
    # --------------------------------------------------

    def transcribe(self, audio_path: str) -> str:
        """Stage 1 (voice only): Whisper speech-to-text."""
        return self.stt.transcribe(audio_path)

    def generate_reply(self, conversation_history: list[dict]) -> str:
        """Stage 2: Qwen3 response. History must already include the latest user message."""
        return self.chatbot.generate_response(conversation_history)

    def speak(self, text: str) -> str:
        """Stage 3: Piper text-to-speech."""
        return str(self.tts.synthesize(text))

### 🖥️ UI layer — Streamlit chat, mic widget, main app

In [ ]:
%%writefile ui/chat.py
import base64
import streamlit as st
from pathlib import Path

PROJECT_ROOT = Path(__file__).resolve().parents[1]


# --------------------------------------------------
# Global styles
# --------------------------------------------------

def _local_css():
    st.markdown("""
    <style>
        /* Centered, spacious content column */
        .block-container {
            max-width: 820px;
            padding-top: 1.2rem;
            padding-bottom: 7rem;
        }
        #MainMenu {visibility: hidden;}
        footer {visibility: hidden;}
        header[data-testid="stHeader"] {background: transparent;}

        /* Message typography */
        .stChatMessage {
            padding: 6px 4px !important;
            background: transparent !important;
        }
        .stChatMessage [data-testid="stMarkdownContainer"] p {
            font-size: 1rem;
            line-height: 1.55;
        }
        .msg-label {
            font-size: 0.72rem;
            font-weight: 600;
            letter-spacing: 0.04em;
            text-transform: uppercase;
            color: #8a8f98;
            margin: 0 0 2px 0;
        }

        /* Compact recorder widget */
        div[data-testid="stAudioInput"] > div {
            min-height: unset;
        }
        section[data-testid="stAudioInput"] {
            border: none !important;
        }

        /* Unified composer container */
        div[data-testid="stVerticalBlockBorderWrapper"] {
            border-radius: 16px !important;
            padding: 12px 14px !important;
            background: rgba(128, 128, 128, 0.04);
        }
        div[data-testid="stVerticalBlockBorderWrapper"] > div {
            gap: 0.35rem !important;
        }

        /* Composer row: input, mic and send share one height & center line.
           Scoped via :has() to the one row containing the text input, so
           welcome chips, level cards and sidebar buttons are untouched.
           NOTE: v1.62 nests the real <button> below a tooltip wrapper, so
           descendant selectors are required (direct-child never matches). */
        div[data-testid="stHorizontalBlock"]:has([data-testid="stTextInput"]) {
            align-items: center;
        }
        [data-testid="stTextInput"] > label {
            display: none;
        }
        [data-testid="stTextInputRootElement"] {
            height: 42px;
        }
        div[data-testid="stHorizontalBlock"]:has([data-testid="stTextInput"]) div.stButton button {
            height: 42px !important;
            min-height: 42px !important;
            max-height: 42px;
            width: 100%;
            max-width: 100%;
            box-sizing: border-box;
            padding: 0 !important;
            margin: 0 auto;
            display: flex !important;
            align-items: center !important;
            justify-content: center !important;
            overflow: hidden;
        }
        div[data-testid="stHorizontalBlock"]:has([data-testid="stTextInput"]) div.stButton button > p {
            margin: 0 !important;
            padding: 0 !important;
            line-height: 1;
            display: flex;
            align-items: center;
            justify-content: center;
        }

        /* Composer buttons round & tidy */
        div.stButton button {
            border-radius: 12px;
            font-size: 1.05rem;
        }
        div.stButton button[kind="primary"] {
            background: #10a37f;
            border-color: #10a37f;
        }
        div.stButton button[kind="primary"]:hover {
            background: #0d8c6d;
            border-color: #0d8c6d;
        }
        div.stButton button[kind="primary"]:disabled {
            opacity: 0.45;
        }
        div.stButton button[kind="secondary"]:hover {
            border-color: #10a37f;
            color: #10a37f;
        }

        /* Example chips on welcome screen */
        .chip-row {display: flex; flex-wrap: wrap; gap: 8px; justify-content: center; margin-top: 10px;}
    </style>
    """, unsafe_allow_html=True)


# --------------------------------------------------
# Single message
# --------------------------------------------------

def _autoplay_audio(path: str):
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    st.markdown(
        '<audio controls autoplay style="width:100%;">'
        f'<source src="data:audio/wav;base64,{b64}" type="audio/wav">'
        "</audio>",
        unsafe_allow_html=True,
    )


def _render_message(message: dict, idx: int, audio_history: dict, autoplay: bool = False):
    role = message["role"]

    if role == "user":
        with st.chat_message("user", avatar="👤"):
            st.markdown('<p class="msg-label">You</p>', unsafe_allow_html=True)
            st.markdown(message["content"])
    else:
        with st.chat_message("assistant", avatar="🇩🇪"):
            st.markdown('<p class="msg-label">LernSathi</p>', unsafe_allow_html=True)
            st.markdown(message["content"])
            audio_path = audio_history.get(idx)
            if audio_path and Path(audio_path).exists():
                if autoplay:
                    _autoplay_audio(audio_path)
                else:
                    with open(audio_path, "rb") as f:
                        st.audio(f.read(), format="audio/wav")


# --------------------------------------------------
# Levels (CEFR)
# --------------------------------------------------

LEVELS = {
    "A1": {"name": "Beginner", "desc": "First words & very simple sentences"},
    "A2": {"name": "Elementary", "desc": "Everyday small talk, simple past"},
    "B1": {"name": "Intermediate", "desc": "Opinions, plans & experiences"},
    "B2": {"name": "Upper-Intermediate", "desc": "Debates & abstract topics"},
    "C1": {"name": "Advanced", "desc": "Fluent, nuanced discussion"},
    "C2": {"name": "Proficient", "desc": "Near-native sophistication"},
}

EXAMPLES = {
    "A1": ["Hallo! Wie geht es dir?", "Ich heiße Anna.", "Ich lerne Deutsch."],
    "A2": ["Was machst du gern am Wochenende?", "Gestern war ich einkaufen.", "Wie ist das Wetter bei dir?"],
    "B1": ["Erzähl mir von deiner Stadt.", "Ich möchte meine Meinung üben.", "Was hast du letztes Jahr gemacht?"],
    "B2": ["Lass uns über soziale Medien diskutieren.", "Ist Fernsehen noch zeitgemäß?", "Welche Rolle spielt Kunst in deinem Leben?"],
    "C1": ["Wie beeinflusst KI die Arbeitswelt?", "Diskutieren wir über Bildungssysteme.", "Erkläre mir ein deutsches Idiom."],
    "C2": ["Ironie im Alltag – Fluch oder Segen?", "Deine Sicht auf moderne Literatur?", "Führe ein Bewerbungsgespräch mit mir."],
}


def render_level_select():
    st.markdown(
        """
        <div style="text-align:center; padding:56px 12px 8px;">
            <div style="font-size:3rem; line-height:1;">🇩🇪</div>
            <h1 style="font-size:1.9rem; font-weight:700; margin:14px 0 4px;">LernSathi</h1>
            <p style="color:#57606a; font-size:1.05rem; margin:0 0 6px;">
                Your German AI Tutor
            </p>
            <p style="color:#57606a; margin:0;">
                Choose your German level to begin.<br>
                The conversation adapts to what you pick.
            </p>
            <hr style="border:none; border-top:1px solid #e6e8eb; width:220px; margin:26px auto;">
        </div>
        """,
        unsafe_allow_html=True,
    )

    rows = [list(LEVELS.items())[i:i + 3] for i in range(0, len(LEVELS), 3)]
    for r, row in enumerate(rows):
        cols = st.columns(3)
        for c, (code, meta) in enumerate(row):
            with cols[c]:
                st.markdown(
                    f"""
                    <div style="text-align:center; padding:14px 6px 4px;">
                        <div style="font-size:1.5rem; font-weight:800;">{code}</div>
                        <div style="font-size:0.9rem; font-weight:600; margin-top:2px;">{meta['name']}</div>
                        <div style="color:#57606a; font-size:0.78rem; margin-top:4px; min-height:2.4em;">
                            {meta['desc']}
                        </div>
                    </div>
                    """,
                    unsafe_allow_html=True,
                )
                if st.button("Start", key=f"level_{code}", use_container_width=True,
                             type="primary" if (r * 3 + c) == 0 else "secondary"):
                    st.session_state.pending_level = code
                    st.rerun()


# --------------------------------------------------
# Welcome screen (English UI, level-aware examples)
# --------------------------------------------------

def _welcome_state(level: str):
    meta = LEVELS[level]
    phrases = EXAMPLES.get(level, [])

    st.markdown(
        f"""
        <div style="text-align:center; padding:56px 12px 8px;">
            <div style="font-size:3rem; line-height:1;">🇩🇪</div>
            <h1 style="font-size:1.9rem; font-weight:700; margin:14px 0 4px;">LernSathi</h1>
            <p style="color:#57606a; font-size:1.05rem; margin:0 0 10px;">
                Your German AI Tutor
            </p>
            <p style="margin:0;">
                <span style="display:inline-block; background:#e7f7f1; color:#0d8c6d;
                border-radius:999px; padding:4px 14px; font-weight:700; font-size:0.85rem;">
                    Level {level} · {meta["name"]}
                </span>
            </p>
            <p style="color:#57606a; margin:16px 0 0;">
                Practice German through natural conversation.<br>
                You can type or speak in German.<br>
                Don't worry about mistakes — LernSathi will help you.
            </p>
            <hr style="border:none; border-top:1px solid #e6e8eb; width:220px; margin:26px auto;">
            <p style="color:#8a8f98; font-size:0.85rem; margin:0 0 4px;">Try saying</p>
        </div>
        """,
        unsafe_allow_html=True,
    )

    cols = st.columns(len(phrases))
    for i, phrase in enumerate(phrases):
        with cols[i]:
            if st.button(phrase, key=f"phrase_{i}", use_container_width=True):
                st.session_state.chat_text_input = phrase
                st.rerun()


# --------------------------------------------------
# Public entry point
# --------------------------------------------------

def render_chat(messages: list, is_processing: bool, audio_history: dict,
                level: str, autoplay_idx: int | None = None,
                stage_status: str | None = None):
    _local_css()

    if not messages and not is_processing:
        _welcome_state(level)
        return

    for idx, msg in enumerate(messages):
        _render_message(msg, idx, audio_history, autoplay=(idx == autoplay_idx))

    # Live pipeline status, rendered inline after the messages so previous
    # chat stays visible while a stage is running.
    if is_processing:
        st.caption(stage_status or "⏳ Working on your message…")

In [ ]:
%%writefile ui/mic_widget.py
import base64
import os

import streamlit.components.v1 as components

_FRONTEND_DIR = os.path.join(os.path.dirname(__file__), "mic_frontend")

_voice_component = components.declare_component("lernsathi_voice", path=_FRONTEND_DIR)


def voice_recorder(action: str | None, cmd_n: int, height: int = 0,
                   key: str = "lernsathi_voice"):
    """
    Inline microphone recorder used by the unified composer.

    action : one-shot command "start" | "send" | "cancel" (None = no command).
             Deduplicated browser-side via cmd_n, so reruns replaying the same
             args never re-trigger a command.
    height : iframe height in px (0 hides the recorder while idle).
    cmd_n  : monotonic counter identifying the current command.

    Returns when the frontend emits:
      {"bytes": wav_bytes, "id": int, "duration_ms": int, "sample_rate": int}
      {"error": str, "id": int}
    Otherwise None.
    """
    val = _voice_component(
        action=action,
        cmd_n=cmd_n,
        height=height,
        key=key,
        default=None,
    )
    if not val:
        return None
    if "error" in val:
        return {"error": str(val["error"]), "id": val.get("id")}
    if "b64" in val:
        return {
            "bytes": base64.b64decode(val["b64"]),
            "id": val["id"],
            "duration_ms": int(val.get("duration_ms", 0)),
            "sample_rate": int(val.get("sample_rate", 48000)),
        }
    return None


In [ ]:
%%writefile ui/mic_frontend/index.html
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8" />
<style>
  html, body { margin: 0; padding: 0; background: transparent; overflow: hidden; }
  #wrap {
    display: flex; align-items: center; gap: 10px;
    height: 60px; padding: 2px 4px;
    font-family: "Source Sans Pro", "Segoe UI", sans-serif;
  }
  #timer {
    font-size: 0.92rem; font-weight: 700; min-width: 46px;
    color: #8a8f98; font-variant-numeric: tabular-nums; flex: none;
  }
  #msg { color: #8a8f98; font-size: 0.85rem; white-space: nowrap; }
  #wave { flex: 1 1 auto; min-width: 0; height: 40px; display: none; }
</style>
</head>
<body>
<div id="wrap">
  <span id="timer">00:00</span>
  <canvas id="wave"></canvas>
  <span id="msg"></span>
</div>
<script>
(function () {
  "use strict";

  var phase = "idle";        // idle | recording
  var targetCmd = null;      // newest queued command: start | send | cancel
  var lastCmdN = -1;         // dedupe: Streamlit may replay identical args
  var chain = Promise.resolve();

  var ctx = null, srcNode = null, procNode = null, analyser = null, mediaStream = null;
  var samples = [], totalLen = 0, capRate = 48000;
  var startedAt = 0, timerId = 0, rafId = 0;
  var timeBuf = null;

  var hist = [], NBARS = 56, dispAmp = 0;

  var elTimer = document.getElementById("timer");
  var elWave = document.getElementById("wave");
  var elMsg = document.getElementById("msg");
  var g2d = elWave.getContext("2d");

  function send(type, data) {
    var msg = Object.assign({ isStreamlitMessage: true, type: type }, data || {});
    window.parent.postMessage(msg, "*");
  }
  function setHeight(h) { send("streamlit:setFrameHeight", { height: h }); }
  function setValue(v) { send("streamlit:setComponentValue", { value: v, dataType: "json" }); }

  function componentReady() {
    send("streamlit:componentReady", { apiVersion: 1 });
    setHeight(0);
  }

  function onRender(event) {
    if (!event.data || event.data.type !== "streamlit:render") return;
    var args = event.data.args || {};
    if (typeof args.height === "number") setHeight(args.height);
    var n = typeof args.cmd_n === "number" ? args.cmd_n : null;
    if (args.action && n !== null && n !== lastCmdN) {
      lastCmdN = n;
      targetCmd = args.action;
      chain = chain.then(exec).catch(function () {});
    }
  }

  function exec() {
    var cmd = targetCmd;
    targetCmd = null;
    if (!cmd) return Promise.resolve();
    if (cmd === "start" && phase === "idle") return startCapture();
    if (cmd === "send" && phase === "recording") return stopCapture(true);
    if (cmd === "cancel" && phase === "recording") return stopCapture(false);
    return Promise.resolve();
  }

  function startCapture() {
    elMsg.textContent = "Waiting for microphone\u2026";
    elWave.style.display = "block";
    setHeight(64);
    return navigator.mediaDevices.getUserMedia({
      audio: { echoCancellation: true, noiseSuppression: true }
    }).then(function (stream) {
      mediaStream = stream;
      ctx = new (window.AudioContext || window.webkitAudioContext)();
      capRate = ctx.sampleRate;
      var resume = ctx.state === "suspended"
        ? ctx.resume().catch(function () {})
        : Promise.resolve();
      return resume.then(function () {
        srcNode = ctx.createMediaStreamSource(stream);
        analyser = ctx.createAnalyser();
        analyser.fftSize = 1024;
        timeBuf = new Uint8Array(analyser.fftSize);
        procNode = ctx.createScriptProcessor(4096, 1, 1);
        samples = [];
        totalLen = 0;
        procNode.onaudioprocess = function (e) {
          var d = e.inputBuffer.getChannelData(0);
          samples.push(new Float32Array(d));
          totalLen += d.length;
        };
        srcNode.connect(analyser);
        srcNode.connect(procNode);
        procNode.connect(ctx.destination);
        startedAt = Date.now();
        timerId = setInterval(tickTimer, 200);
        elMsg.textContent = "";
        phase = "recording";
        tickTimer();
        loop();
      });
    }).catch(function (err) {
      fail(String((err && err.message) || err || "Microphone unavailable"));
    });
  }

  function stopCapture(emit) {
    clearInterval(timerId); timerId = 0;
    if (rafId) { cancelAnimationFrame(rafId); rafId = 0; }
    var durationMs = Date.now() - startedAt;
    try { if (procNode) procNode.disconnect(); } catch (e) {}
    try { if (srcNode) srcNode.disconnect(); } catch (e) {}
    try { if (mediaStream) mediaStream.getTracks().forEach(function (t) { t.stop(); }); } catch (e) {}
    try { if (ctx) ctx.close(); } catch (e) {}
    ctx = srcNode = procNode = analyser = mediaStream = null;
    phase = "idle";
    if (emit) {
      if (totalLen > 0) {
        var wav = encodeWav(mergeSamples(), capRate);
        setValue({
          b64: bufToB64(wav),
          id: Date.now(),
          duration_ms: durationMs,
          sample_rate: capRate
        });
      } else {
        setValue({ error: "No audio captured.", id: Date.now() });
      }
    }
    samples = []; totalLen = 0; hist = []; dispAmp = 0;
    uiRecordingOff();
  }

  function fail(message) {
    if (phase === "recording") stopCapture(false);
    setValue({ error: message, id: Date.now() });
    uiRecordingOff();
  }

  function uiRecordingOff() {
    elWave.style.display = "none";
    elTimer.textContent = "00:00";
    elMsg.textContent = "";
    g2d.clearRect(0, 0, elWave.width, elWave.height);
    setHeight(0);
  }

  function tickTimer() {
    var s = Math.floor((Date.now() - startedAt) / 1000);
    elTimer.textContent =
      ("0" + Math.floor(s / 60)).slice(-2) + ":" + ("0" + (s % 60)).slice(-2);
  }

  function loop() {
    if (phase !== "recording") return;
    draw();
    rafId = requestAnimationFrame(loop);
  }

  function draw() {
    var dpr = window.devicePixelRatio || 1;
    var W = elWave.clientWidth, H = elWave.clientHeight;
    if (!W || !H) return;
    if (elWave.width !== Math.round(W * dpr)) elWave.width = Math.round(W * dpr);
    if (elWave.height !== Math.round(H * dpr)) elWave.height = Math.round(H * dpr);
    g2d.setTransform(dpr, 0, 0, dpr, 0, 0);
    g2d.clearRect(0, 0, W, H);

    var peak = 0.03;
    if (analyser) {
      analyser.getByteTimeDomainData(timeBuf);
      for (var i = 0; i < timeBuf.length; i++) {
        var v = Math.abs(timeBuf[i] - 128) / 128;
        if (v > peak) peak = v;
      }
    }
    dispAmp = Math.max(peak, dispAmp * 0.86);
    hist.push(dispAmp);
    while (hist.length > NBARS) hist.shift();

    var slot = W / NBARS;
    var bw = Math.max(2, slot * 0.55);
    g2d.fillStyle = "#e5484d";
    for (var j = 0; j < hist.length; j++) {
      var bh = Math.max(2, Math.min(1, hist[j]) * (H - 4));
      g2d.fillRect(j * slot + (slot - bw) / 2, (H - bh) / 2, bw, bh);
    }
  }

  function mergeSamples() {
    var out = new Float32Array(totalLen), off = 0;
    for (var i = 0; i < samples.length; i++) {
      out.set(samples[i], off);
      off += samples[i].length;
    }
    return out;
  }

  function encodeWav(s, rate) {
    var buf = new ArrayBuffer(44 + s.length * 2);
    var v = new DataView(buf);
    function ws(o, t) { for (var i = 0; i < t.length; i++) v.setUint8(o + i, t.charCodeAt(i)); }
    ws(0, "RIFF"); v.setUint32(4, 36 + s.length * 2, true);
    ws(8, "WAVE"); ws(12, "fmt ");
    v.setUint32(16, 16, true);
    v.setUint16(20, 1, true);
    v.setUint16(22, 1, true);
    v.setUint32(24, rate, true);
    v.setUint32(28, rate * 2, true);
    v.setUint16(32, 2, true);
    v.setUint16(34, 16, true);
    ws(36, "data"); v.setUint32(40, s.length * 2, true);
    var o = 44;
    for (var i = 0; i < s.length; i++) {
      var x = Math.max(-1, Math.min(1, s[i]));
      v.setInt16(o, x < 0 ? x * 0x8000 : x * 0x7fff, true);
      o += 2;
    }
    return buf;
  }

  function bufToB64(buf) {
    var u8 = new Uint8Array(buf), s = "", chunk = 0x8000;
    for (var i = 0; i < u8.length; i += chunk) {
      s += String.fromCharCode.apply(null, u8.subarray(i, i + chunk));
    }
    return btoa(s);
  }

  window.addEventListener("message", onRender);
  if (document.readyState === "complete") componentReady();
  else window.addEventListener("load", componentReady);
})();
</script>
</body>
</html>


### ▶️ `app.py` — the Streamlit entry point

In [ ]:
%%writefile app.py
import os
import uuid
import streamlit as st

from ui.chat import render_chat, render_level_select, _local_css, LEVELS
from ui.mic_widget import voice_recorder
from services.conversation_service import ConversationService


# --------------------------------------------------
# Page configuration
# --------------------------------------------------

st.set_page_config(
    page_title="LernSathi",
    page_icon="🇩🇪",
    layout="wide",
)

_local_css()


# --------------------------------------------------
# Session state (deterministic across reruns)
# --------------------------------------------------

_defaults = {
    "level": None,             # selected CEFR level (A1–C2); None = not chosen yet
    "messages": [],            # single source of truth for the chat
    "audio_history": {},       # assistant msg index -> tts wav path
    "last_audio_id": None,     # dedupe guard for processed voice messages
    "composer_mode": "text",   # "text" | "recording"
    "mic_cmd": "",             # last one-shot command for the recorder
    "mic_cmd_n": 0,            # monotonic counter -> browser-side dedupe
    "send_in_flight": False,   # recording sent, waiting for the audio bytes
    "is_processing": False,    # global pipeline lock
    "processing_stage": "",    # "" | "transcribe" | "respond"
    "pending_audio_path": "",  # saved recording awaiting Whisper
    "pipeline_error": "",      # friendly error shown once on next paint
}
for key, val in _defaults.items():
    if key not in st.session_state:
        st.session_state[key] = val

# Clear widget-bound keys BEFORE their widgets are instantiated this run
if st.session_state.pop("clear_chat_input", False):
    st.session_state.chat_text_input = ""


# --------------------------------------------------
# Load AI models once (Whisper small / qwen3:1.7b / Piper)
# --------------------------------------------------

@st.cache_resource
def load_service():
    return ConversationService()


service = load_service()

# Keep the cached service in sync with the selected level
if st.session_state.level is not None:
    service.set_level(st.session_state.level)


# --------------------------------------------------
# Helpers
# --------------------------------------------------

STAGE_STATUS = {
    "transcribe": "🎤 Transcribing your message…",
    "respond": "🤔 LernSathi is responding…",
}


def paint():
    """Redraw the conversation from session state."""
    autoplay_idx = st.session_state.pop("autoplay_idx", None)
    render_chat(
        messages=st.session_state.messages,
        is_processing=st.session_state.is_processing,
        audio_history=st.session_state.audio_history,
        level=st.session_state.level,
        autoplay_idx=autoplay_idx,
        stage_status=STAGE_STATUS.get(st.session_state.processing_stage),
    )


def reset_conversation():
    """Clear the chat history (keeps the selected level)."""
    st.session_state.messages = []
    st.session_state.audio_history = {}
    st.session_state.last_audio_id = None
    st.session_state.composer_mode = "text"
    st.session_state.mic_cmd = ""
    st.session_state.send_in_flight = False
    st.session_state.is_processing = False
    st.session_state.processing_stage = ""
    st.session_state.pending_audio_path = ""
    st.session_state.pipeline_error = ""
    st.session_state.chat_text_input = ""


def _mic_command(name: str):
    """Issue a one-shot command to the recorder component."""
    st.session_state.mic_cmd_n += 1
    st.session_state.mic_cmd = name


def _handle_recorder(recorder):
    """Consume an emitted recording / error from the recorder component."""
    if not recorder:
        return
    st.session_state.send_in_flight = False
    st.session_state.composer_mode = "text"

    rid = recorder.get("id")
    if "error" in recorder:
        if rid is not None and rid != st.session_state.last_audio_id:
            st.session_state.last_audio_id = rid
            _fail(f"Microphone error ({recorder['error']}).",
                  Exception(recorder["error"]))
        st.rerun()

    if rid is not None and rid != st.session_state.last_audio_id:
        st.session_state.last_audio_id = rid
        rec_dir = os.path.join("audio", "input")
        os.makedirs(rec_dir, exist_ok=True)
        rec_path = os.path.join(rec_dir, f"user_recording_{uuid.uuid4().hex}.wav")
        with open(rec_path, "wb") as f:
            f.write(recorder["bytes"])
        st.session_state.pending_audio_path = rec_path
        _start_pipeline("transcribe")


def _fail(message: str, exc: Exception):
    st.error(f"{message} Please try again.")
    print(f"[LernSathi] {type(exc).__name__}: {exc}")


def _rollback_unpaired_user():
    """If a user message was added but no reply followed, remove it."""
    if st.session_state.messages and st.session_state.messages[-1]["role"] == "user":
        st.session_state.messages.pop()


def _finish():
    st.session_state.is_processing = False
    st.session_state.processing_stage = ""
    st.session_state.pending_audio_path = ""
    st.session_state.clear_chat_input = True


def _friendly_error(e: Exception) -> str:
    print(f"[LernSathi] {type(e).__name__}: {e}")
    if isinstance(e, ValueError):
        return f"{e} Please try again."
    if "piper" in str(e).lower():
        return "I couldn't generate the voice response. Please try again."
    return "The tutor is temporarily unavailable. Please try again."


def _start_pipeline(stage: str):
    """Accept a new request and lock the UI until the pipeline completes."""
    st.session_state.processing_stage = stage
    st.session_state.is_processing = True
    st.rerun()


def _drive_pipeline():
    """
    Run the next pending pipeline stage.

    Called AFTER this run's UI has been drawn, so each finished stage becomes
    visible (via st.rerun at the end of every run) before the next one starts:

    TEXT :  Qwen + Piper together            (Whisper skipped entirely)
    VOICE:  Whisper, then Qwen + Piper       (transcription shown as user bubble)

    The reply text and its audio are always shown as ONE unit once both are
    ready — only Whisper is split out so the transcription appears instantly.
    """
    stage = st.session_state.processing_stage
    if not stage or not st.session_state.is_processing:
        return

    try:
        if stage == "transcribe":
            # Stage 1: Whisper -> immediately show the transcription.
            user_text = service.transcribe(st.session_state.pending_audio_path)
            if not user_text or not user_text.strip():
                raise ValueError("The recording could not be understood.")
            st.session_state.messages.append({"role": "user", "content": user_text})
            st.session_state.processing_stage = "respond"

        elif stage == "respond":
            # Stage 2: Qwen reply + Piper voice prepared together and shown
            # as one message (text + audio) when both are ready.
            reply = service.generate_reply(st.session_state.messages)
            audio_path = service.speak(reply)
            st.session_state.messages.append({"role": "assistant", "content": reply})
            last_idx = len(st.session_state.messages) - 1
            st.session_state.audio_history[last_idx] = audio_path
            st.session_state.autoplay_idx = last_idx
            st.session_state.processing_stage = ""

    except Exception as e:
        if stage == "respond":
            _rollback_unpaired_user()
        st.session_state.pipeline_error = _friendly_error(e)
        st.session_state.processing_stage = ""

    finally:
        if not st.session_state.processing_stage:
            _finish()
        st.rerun()


# --------------------------------------------------
# Sidebar
# --------------------------------------------------

with st.sidebar:
    st.markdown("### 🇩🇪 LernSathi")
    st.caption("German AI Conversation Tutor")

    if st.button("＋ New conversation", use_container_width=True):
        reset_conversation()
        st.rerun()

    st.divider()

    st.markdown("**Current level**")
    if st.session_state.level is None:
        st.caption("Not selected yet")
    else:
        meta = LEVELS[st.session_state.level]
        st.markdown(
            f"<span style='display:inline-block; background:#e7f7f1; color:#0d8c6d;"
            f"border-radius:999px; padding:3px 12px; font-weight:700; font-size:0.85rem;'>"
            f"{st.session_state.level} · {meta['name']}</span>",
            unsafe_allow_html=True,
        )
        if st.button("🎚 Change level", use_container_width=True):
            reset_conversation()
            st.session_state.level = None
            st.rerun()

    st.divider()

    st.markdown("**Current session**")
    st.caption("German practice · not saved anywhere")
    st.markdown(f"**{len(st.session_state.messages)} messages** in this conversation")

    st.divider()

    st.markdown("**About**")
    st.caption(
        "Practice German through AI-powered "
        "conversation. Everything runs locally."
    )


# --------------------------------------------------
# Main area
# --------------------------------------------------

if st.session_state.level is None:
    # Level-first flow: no level chosen -> show picker, hide chat
    render_level_select()

    pending = st.session_state.pop("pending_level", None)
    if pending:
        st.session_state.level = pending
        service.set_level(pending)
        reset_conversation()
        st.rerun()
else:
    paint()

# Friendly pipeline error from the previous stage run (shown once).
pipeline_error = st.session_state.pop("pipeline_error", None)
if pipeline_error:
    st.error(pipeline_error)


# --------------------------------------------------
# Unified composer — ONE component, two states
#
# TEXT      : [ Write in German...        ][ 🎤 ][ ➤ ]
# RECORDING : [ 00:07  ~~~live waveform~~~ ][ ✕  ][ ➤ ]
#             Send stops the recording AND submits it.
# --------------------------------------------------

if st.session_state.level is not None:

    processing = st.session_state.is_processing
    recording_mode = st.session_state.composer_mode == "recording"

    with st.container(border=True):

        if not recording_mode:
            col_a, col_b, col_c = st.columns([8, 0.55, 0.55])

            with col_a:
                typed = st.text_input(
                    "Message",
                    placeholder="Write in German...",
                    key="chat_text_input",
                    label_visibility="collapsed",
                    disabled=processing,
                )

            with col_b:
                if st.button("🎤", disabled=processing, use_container_width=True,
                             key="btn_mic", help="Record a voice message"):
                    _mic_command("start")
                    st.session_state.composer_mode = "recording"
                    st.rerun()

            with col_c:
                text_send_clicked = st.button(
                    "➤", type="primary",
                    disabled=processing or not typed,
                    use_container_width=True, key="btn_send",
                    help="Send message",
                )

            # Accept the message and hand over to the stage driver:
            # the user bubble paints on the next rerun, before Qwen runs.
            if text_send_clicked and typed and not processing:
                st.session_state.messages.append({"role": "user", "content": typed})
                _start_pipeline("respond")

        else:
            col_a, col_b, col_c = st.columns([8, 0.55, 0.55])

            with col_a:
                st.caption("🔴 Recording")

            with col_b:
                if st.button("✕", disabled=processing or st.session_state.send_in_flight,
                             use_container_width=True, key="btn_cancel_rec",
                             help="Discard recording"):
                    _mic_command("cancel")
                    st.session_state.composer_mode = "text"
                    st.session_state.send_in_flight = False
                    st.rerun()

            with col_c:
                if st.button("➤", type="primary", disabled=processing,
                             use_container_width=True, key="btn_send_rec",
                             help="Stop and send"):
                    _mic_command("send")
                    st.session_state.send_in_flight = True
                    st.rerun()

            # Live timer + waveform live inside this iframe; on "send" it
            # stops capturing and emits the WAV bytes back to Python.
            recorder_result = voice_recorder(
                action=st.session_state.mic_cmd or None,
                cmd_n=st.session_state.mic_cmd_n,
                height=64,
            )
            _handle_recorder(recorder_result)

    # Run the next pending pipeline stage (Whisper / Qwen / Piper) AFTER the
    # UI above has been drawn, so each finished stage is painted before the
    # next one blocks this run. No global spinner — stages stay visible.
    _drive_pipeline()

In [ ]:
#@title Step 5 — Download the Piper voice · de_DE thorsten medium (~65 MB)
!mkdir -p models/tts
!wget -qc https://huggingface.co/rhasspy/piper-voices/resolve/main/de/de_DE/thorsten/medium/de_DE-thorsten-medium.onnx -O models/tts/de_DE-thorsten-medium.onnx
!wget -qc https://huggingface.co/rhasspy/piper-voices/resolve/main/de/de_DE/thorsten/medium/de_DE-thorsten-medium.onnx.json -O models/tts/de_DE-thorsten-medium.onnx.json
!ls -lh models/tts/

In [ ]:
#@title Step 6 — Start the Ollama server (background daemon)
import subprocess
import time
import urllib.request


def server_alive():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        return True
    except Exception:
        return False


if server_alive():
    print("Ollama server already running ✔")
else:
    # start_new_session keeps the daemon alive after this cell finishes
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    for _ in range(60):
        if server_alive():
            break
        time.sleep(1)
    else:
        raise RuntimeError("Ollama server did not come up within 60 s")

print("Ollama server ready ✔  (http://127.0.0.1:11434)")

In [ ]:
#@title Step 7 — Pull the Qwen3 1.7B model into Ollama (~1.4 GB, one-time)
!ollama pull qwen3:1.7b
!ollama list

## ✅ Smoke test — hear the whole pipeline before opening the UI

This loads **Whisper small** (~460 MB, auto-download) and runs a full round-trip:
LLM reply → Piper voice → Whisper transcription of that voice.

In [ ]:
#@title Test the full pipeline: LLM → TTS → STT round-trip
from IPython.display import Audio, display

from ai.llm.model import LEVEL_PROMPTS
from services.conversation_service import ConversationService

LEVEL = "A1" #@param ["A1", "A2", "B1", "B2", "C1", "C2"]

print("Loading models (first run downloads Whisper small)...\n")
service = ConversationService(level=LEVEL)
print(f"Tutor level: {service.level}")

# 1) LLM — reply adapts to the selected level
reply = service.generate_reply([{"role": "user", "content": "Hallo, wer bist du?"}])
print("LernSathi >", reply, "\n")

# 2) Switch level on the fly — only the system prompt changes, no reload
service.set_level("C2")
reply_c2 = service.generate_reply([{"role": "user", "content": "Erzähl mir von deiner Stadt."}])
print(f"LernSathi [C2] >", reply_c2, "\n")
service.set_level(LEVEL)

# 3) TTS — play it right here
wav_path = service.speak("Hallo! Ich bin LernSathi, dein deutscher Sprachlehrer.")
print("TTS audio ->", wav_path)
display(Audio(wav_path))

# 4) STT — feed the generated audio back through Whisper
print("Whisper heard >", service.transcribe(wav_path))


In [ ]:
# #@title Optional — chat with the tutor directly in this notebook
# LEVEL = "A1" #@param ["A1", "A2", "B1", "B2", "C1", "C2"]

# if service.level != LEVEL:
#     service.set_level(LEVEL)   # instant switch, models stay loaded

# history = []
# print(f'Tutor level: {service.level}. Type in German. Type "quit" to exit.\n')
# while True:
#     try:
#         user = input("Du > ").strip()
#     except (KeyboardInterrupt, EOFError):
#         break
#     if not user or user.lower() in {"quit", "exit", "q"}:
#         break
#     history.append({"role": "user", "content": user})
#     reply = service.generate_reply(history)
#     history.append({"role": "assistant", "content": reply})
#     print(f"LernSathi > {reply}\n")


## 🌐 Launch the Streamlit UI (same experience as local)

The app starts on port 8501 and is exposed via a free Cloudflare quick tunnel.
**Keep the notebook tab open while demoing** — closing/disconnecting the runtime kills the tunnel.

In [ ]:
#@title Step 8 — Launch Streamlit + public HTTPS tunnel 🚀
import re
import subprocess
import time

# Clean up any previous instances
subprocess.run("pkill -f 'streamlit run' || true", shell=True)
subprocess.run("pkill -f cloudflared || true", shell=True)

st_log = open("streamlit.log", "w")
subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port=8501",
        "--server.address=127.0.0.1",
        "--server.headless=true",
        "--browser.gatherUsageStats=false",
    ],
    stdout=st_log,
    stderr=subprocess.STDOUT,
)
print("Streamlit starting on port 8501 ...")
time.sleep(8)

cf_log = open("cloudflared.log", "w")
subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8501", "--no-autoupdate"],
    stdout=cf_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
    if m:
        public_url = m.group(0)
        break

print()
if public_url:
    print("=" * 62)
    print(f"  🇩🇪 LernSathi is LIVE ->  {public_url}")
    print("=" * 62)
    print("Open the link, click 'Allow' when the browser asks for the")
    print("microphone, hit 🎤 and speak German!")
else:
    print("Could not find the tunnel URL yet. Log output:")
    print(open("cloudflared.log").read())

## 🛠️ Notes & Troubleshooting

| Symptom | Fix |
|---|---|
| First answer takes ~30–60 s | Normal — Qwen3 warms up on its first request; afterwards replies are fast |
| Microphone doesn't ask for permission | You must be on the `https://…trycloudflare.com` URL (not localhost); re-run **Step 8** to get a fresh tunnel |
| Tunnel stopped responding | The Colab VM idled out — re-run **Step 8** only (models stay loaded if the runtime wasn't deleted) |
| Everything broke / kernel restarted | Disk persists across kernel restarts: re-run **Step 6** onward. If the VM itself was recycled: **Runtime ▸ Run all** |
| No GPU available | Fine — Whisper falls back to CPU and Qwen3:1.7B runs on CPU too, just noticeably slower |
| Want a fresh conversation or another difficulty | Click **“＋ New conversation”** (same level) or **“🎚 Change level”** (back to A1–C2 picker) in the app sidebar |

**Demo tip:** run Steps 1–7 before your audience arrives; during the demo just fire Step 8, open the link, and talk. 🎤